<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day17_practice1_%EC%96%B4%ED%85%90%EC%85%98_%EC%86%90%EA%B3%84%EC%82%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from torch._C import device
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import urllib.request, os
from collections import Counter # Counter=단어 빈도 세기

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
# 셀 1. NSMC - 네이버 영화 리뷰
URL = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
urllib.request.urlretrieve(URL, "ratings_train.txt")
print("ratings_train.txt 다운로드 완료")
texts, labels = [], []
with open("ratings_train.txt", encoding = "utf-8") as f:
  next(f) # 첫 줄(헤더 id document label)을 한 번 읽어 버림
  for line in f:
    parts = line.strip().split("\t")
    if len(parts) == 3 and parts[1]:
      texts.append(parts[1])
      labels.append(int(parts[2]))
print(f"전체 리뷰 {len(texts):,}개 | 긍정 {sum(labels):,}/부정 {len(labels)-sum(labels):,}")
print("샘플:", texts[0], "→", "긍정" if labels[0] else "부정")

N = 30000
texts, labels = texts[:N], labels[:N] # 3만개만 슬라이싱

ratings_train.txt 다운로드 완료
전체 리뷰 149,995개 | 긍정 74,825/부정 75,170
샘플: 아 더빙.. 진짜 짜증나네요 목소리 → 부정


In [9]:
# 셀 2. vocab - '빈도 상위'만
counter = Counter(tok for t in texts for tok in t.split()) # {단어: 등장횟수}
print(f"고유 어절 수: {len(counter):,}개")
print("최다 빈도:", counter.most_common(5))

VOCAB_SIZE = 15000
vocab = {"<pad>": 0, "<unk>": 1}
for tok, _ in counter.most_common(VOCAB_SIZE - 2):
  vocab[tok] = len(vocab) # 번호 부여

MAX_LEN = 20
def encode(text):
  ids = [vocab.get(t, 1) for t in text.split()][:MAX_LEN] # 단어를 번호로 바꾸고 너무 길면 자른다
  return ids + [0] * (MAX_LEN - len(ids)) # 너무 짧으면 뒤를 0으로 채운다
print(f"\n원문: {texts[0]!r}")
print(f"[변수 확인] encode(texts[0]) = {encode(texts[0])}") # 첫 리뷰 문장의 단어를 번호로 바꾼거

X = torch.tensor([encode(t) for t in texts])
y = torch.tensor(labels, dtype = torch.float32).reshape(-1,1)
lengths = (X != 0).sum(dim=1) # 문장 실제 길이
len_test = lengths[n_train:].to(device) # 테스트 데이터의 실제 문장 길이

print(f"\nX.shape = {X.shape}")
print(f"y.shape = {y.shape}")

print(f"X[0] = {X[0]}")
print(f"y[0] = {y[0]}")

# 학습/평가 분리
n_train = int(N * 0.9)
train_loader = DataLoader(TensorDataset(X[:n_train], y[:n_train], lengths[:n_train]), batch_size = 256, shuffle=True) # 앞 90% # [256, 20] :문장, 단어
X_test, y_test = X[n_train:].to(device), y[n_train:].to(device) # 뒤 10%

print(f"X_test.shape = {X_test.shape}, y_test.shape = {y_test.shape}")

고유 어절 수: 98,463개
최다 빈도: [('영화', 2241), ('너무', 1602), ('정말', 1566), ('진짜', 1191), ('이', 1021)]

원문: '아 더빙.. 진짜 짜증나네요 목소리'
[변수 확인] encode(texts[0]) = [51, 1, 5, 10248, 1557, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

X.shape = torch.Size([30000, 20])
y.shape = torch.Size([30000, 1])
X[0] = tensor([   51,     1,     5, 10248,  1557,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0])
y[0] = tensor([0.])
X_test.shape = torch.Size([3000, 20]), y_test.shape = torch.Size([3000, 1])


In [7]:
# 셀 3. 모델 - 평균 대신 LSTM의 '마지막 기억', '순서대로 읽은 꼴의 기억h'이 문장 요약이라,
# 어디가 끝인지(진짜 마지막 위치)를 gather로 정확히 집어야 한다

# 평균은 순서를 버려서 "재미가 없지 않다"의 "재미가 있지 않다"가 같은 결과
# LSTM은 단어를 하나씩 순서대로 읽으며 기억(h)을 갱신하므로, 순서를 이해한다.

class LSTMSentiment(nn.Module):
  def __init__(self):
    super().__init__()
    self.emb = nn.Embedding(len(vocab), 64, padding_idx=0) # 임베딩 (15000, 64)
    self.lstm = nn.LSTM(64, 64, batch_first=True) # 입력: 64차원 단어를 읽어, 은닉: 64차원 기억h으로 문장 요약, {배치, 단어수, 차원}
    self.fc = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32,1), nn.Sigmoid()) # 분류기, Sigmoid: 최종 점수를 0-1 확률로
  def forward(self, x, lengths):
    out, _ = self.lstm(self.emb(x)) # out: {B, 20, 64} 모든 걸음의 h
    idx = (lengths - 1).clamp(min=0).view(-1,1,1).expand(-1,1,64) # 각 문장의 진짜 마지막 위치( 마지막 단어의 인덱스)를 gather가 쓸 수 있는 모양으로 만든다.
    last_h = out.gather(1, idx).squeeze(1) # (B,1,64) → (B,64) 문장벡터, 문장마다 idx가 가리키는 위치의 h를 꺼낸다.
    return self.fc(last_h)

model = LSTMSentiment().to(device)
loss_fn = nn.BCELoss()
opt = torch.optim.Adam(model.parameters(), lr=0.002)


In [12]:
# 셀 4. 학습
EPOCHS = 5
for epoch in range(EPOCHS):
  model.train()
  for xb, yb, ib in train_loader:   # 배치마다 (입력, 정답, 길이)
    xb, yb, ib = xb.to(device), yb.to(device), ib.to(device) # {B,20} {B,1} {B,3}
    loss = loss_fn(model(xb, ib), yb)
    opt.zero_grad()
    loss.backward()
    opt.step()

  model.eval()
  with torch.no_grad():
    acc = ((model(X_test, len_test) > 0.5) == y_test.bool()).float().mean().item()
  print(f"epoch {epoch+1}/{EPOCHS} | 테스트 정확도 {acc:.4f}")

epoch 1/5 | 테스트 정확도 0.7263
epoch 2/5 | 테스트 정확도 0.7357
epoch 3/5 | 테스트 정확도 0.7310
epoch 4/5 | 테스트 정확도 0.7347
epoch 5/5 | 테스트 정확도 0.7297


In [13]:
# 셀 5. 단어의 순서를 기억한다.
def predict(sentence):
  model.eval()
  x = torch.tensor([encode(sentence)]).to(device) # [(1,20)]
  i = (x != 0).sum(dim=1) # 진짜 단어 길이
  with torch.no_grad():
    p = model(x, i).item()
  return p #확률값

pairs = [
    ("지루하다 하지만 결말은 최고", "최고 하지만 결말은 지루하다"),
    ("재미가 없지 않다", "재미가 있지 않다"),
]
print("[순서만 다른 문장 쌍]")
for a, b in pairs:
  pa, pb = predict(a), predict(b)
  print(f" {a!r}: {pa:.0%}")
  print(f" {b!r}: {pb:.0%}\n")

[순서만 다른 문장 쌍]
 '지루하다 하지만 결말은 최고': 99%
 '최고 하지만 결말은 지루하다': 0%

 '재미가 없지 않다': 1%
 '재미가 있지 않다': 5%

